# Data Description
- Author: 詹明翰、謝子詮、曾翊涵（共同作者不等於已完成個別工作）
- Date first created: 20261126
- Background: Dataset B1 年度設籍人口結構：population_density、elderly_ratio、children_ratio及人口月和、平均、有效月份、原始代碼／名稱、固定面積與證據狀態。作D原版事故分群後核心外部環境比較，不自動加入事故組成分群。B1無S0–S3重複列，事故規格不改人口。
- Unit of observation: 固定研究行政里 × year；village_id＋year主鍵，912列。
- Sample period: 2024與2025分年，各12個月；原始2023及2026不納入特徵。
- Known issues: 設籍人口非實住／日間／通勤／道路使用人口或交通暴露，人口密度非流量或VKT，事故地非當事人住地，地區年齡非涉事者年齡。固定456里是共同框架，不認證各月歷史界線；月基準日及精確年齡慣例仍有限制。只使用計，男／女僅驗證不重複加總；月存量和非期間不同個人總數。缺月或人口欄缺失不補0、不以少月份平均冒充完整期間，受影響特徵NA；分母0比例NA。極端值只診斷不刪除、不縮尾／對數／標準化／補值。糖廍provisional保留但正常納入，公館不因事故零觀測排除。生態關聯不支持個人或因果推論，不建立居民事故風險或事故／人口率。成員交付確認及DA／PM驗收待完成。 原始月存量另有2025-06龍福5140→6215、幸福5537→4471跳動，月份及代碼完整；原因與歷史空間可比性未認證，詳處理Notebook月序列，保留原值待學生審查。
- Definition for each variable: 下方字典逐一列出全部46欄之dtype／定義／單位／來源／缺失規則。年齡65–99加100歲以上為高齡、0–14為兒童，15–64只作上游品質驗證。
- Data path: `data/processed/taipei_dataset_b1_annual.csv`
- Source data: 原人口CP950資料、既有crosswalk、正式里主表與必要驗證證據，11項人口來源／helper及固定SHA見POPULATION_INPUTS；8項A／C僅供合併鍵驗證，不作人口數值輸入。
- Processing code: [處理Notebook](../code/prepare_taipei_dataset_b1.ipynb)；[helper](../code/unit24_population_support.py)。Producer SHA `27ef88b179ae346479cab1d53b35c41a537cf7383d03216d4c4923c32eaa7ff0`。
- Role: external_environment_population
- Verification status: 正式CSV 912×46、358,874 bytes，先由處理Notebook精確讀回及直接24月對年度加總核對；本文件下方Assertions保存獨立乾淨kernel實際結果，技術通過不等於人工驗收。
- SHA-256: `d4ad76e82ad6f00d5b146c00c0e065d2095cd9fd36fe6bbbad5feb40705ae795`
- Schema: `data/processed/taipei_dataset_b1_annual_schema.json`；SHA `89bb5ace9a7abf9bcedcc1ff0d6b2f91d45578f7c6cfe20a685bd6802d8fa7ca`。
- Downstream analysis input: `data/processed/taipei_dataset_b1_annual.csv`（與Data path相同）

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。Deliverables：本資料／schema／獨立說明及共同處理。Actual Contribution：Codex協助，成員實際工作及交付審查待完成。


## Packages、來源與正式版本

In [1]:
from pathlib import Path
import sys,json,hashlib,platform
import pandas as pd
import numpy as np
from IPython.display import display,Markdown
for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
    if (p/'AGENTS.md').is_file() and (p/'templates').is_dir():
        PROJECT_ROOT=p;break
else:raise FileNotFoundError('請從研究工作包內執行')
assert Path(sys.executable).resolve()==(PROJECT_ROOT/'.venv/Scripts/python.exe').resolve()
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for part in iter(lambda:stream.read(4*1024*1024),b''):h.update(part)
    return h.hexdigest()
assert sha(PROJECT_ROOT/'data/code/unit24_population_support.py')=='ce49653ccfec3fc9000f56f58076021ac906764806780d0c7a9e44b4aa2060a4'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
import unit24_population_support as u
POPULATION_INPUTS={'data/raw/112迄今各里人口數按年齡分.csv': 'a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641', 'data/raw/taipei_population_age_raw_documentation.ipynb': '8b7b5376b6e79821c0cdaffd7f17e97f9e466bb3933d6b77552cf48846c29f53', 'data/processed/village_crosswalk.csv': 'dd9e762419c302cf5b4bb19daad2673d7d54d0bb750204e7e33ecbf1a9983f69', 'data/processed/village_crosswalk_documentation.ipynb': '768ebaac028b635c93b9c768efefdb504c902d49da586e5f33180a7c9855fce8', 'data/processed/taipei_village_base.csv': 'db0a225df3568ddb2a47eab1e1d28843e1594dec37956c57eea926e444968209', 'data/processed/taipei_village_base_schema.json': 'a7e5e3b4d77ab506736ecc940e26a87df6cace3a738cbdaeef1e835e6bed7070', 'data/processed/taipei_village_base_documentation.ipynb': 'a843d587788b4ea6a943cc05d9413908cf8b66b2f73f9fdbec01e2ab52edd535', 'data/code/taipei_population_validation.ipynb': 'e26ff652ccfa2113db041322a99a000d8cb6d71acccdab66c35448fcddaab7d8', 'data/code/taipei_data_integration_readiness.ipynb': '6048ff52d7653c9ec9d4b32cf2b78da2fb1b22dc0db0164cc48c4d2d98dd338b', 'data/code/prepare_taipei_village_base_and_crash_spatial_assignment.ipynb': 'ef8a3d21f38ecb902f9495720b0094363ebd57cf444b837176fe2b815992abc4', 'data/code/unit24_population_support.py': 'ce49653ccfec3fc9000f56f58076021ac906764806780d0c7a9e44b4aa2060a4'}
HANDOFF_ONLY_INPUTS={'data/processed/taipei_dataset_a_annual.csv': '4e264176c970263777eb4c564f427e1f839a45b31552a37bc4bdbfbed6b09422', 'data/processed/taipei_dataset_a_annual_schema.json': '4d0c5b7b307b1cca4261a9595cd1ca894f549d88e06bed1b0b071dcf7c40fb21', 'data/processed/taipei_dataset_a_pooled.csv': '081c7dd42c94e16a5684f27edaebd310a9c556bae7460783dd87fde8d26ea46a', 'data/processed/taipei_dataset_a_pooled_schema.json': '7878146fac39d4bfabee965b6001d0cf5e101ca1d76778599fec4b480ad59b5a', 'data/processed/taipei_dataset_c_annual.csv': '9ab1970ed0f790ebdf31e1726094493305da00ac4efaf84b8a554f7debfa560e', 'data/processed/taipei_dataset_c_annual_schema.json': '4a8151a9a5dcd716cc03312b303c4455cb47210192c43b86cf3a0e06d268cb6f', 'data/processed/taipei_dataset_c_pooled.csv': 'caceea61ff906b3e2f0b4796a31af3373a929307a0c7f7094fb9373794f76bc6', 'data/processed/taipei_dataset_c_pooled_schema.json': '44baac19975650d792e23e81c5463b84e00f299c9c606ef0f95abf8b3756d3ab'}
u.verify(PROJECT_ROOT,POPULATION_INPUTS)
u.verify(PROJECT_ROOT,HANDOFF_ONLY_INPUTS)
print('Python',platform.python_version(),'人口來源／helper hash',len(POPULATION_INPUTS),'僅交付鍵檢查來源hash',len(HANDOFF_ONLY_INPUTS))

EXPECTED_OUTPUTS={'data/processed/taipei_dataset_b1_annual.csv': 'd4ad76e82ad6f00d5b146c00c0e065d2095cd9fd36fe6bbbad5feb40705ae795', 'data/processed/taipei_dataset_b1_annual_schema.json': '89bb5ace9a7abf9bcedcc1ff0d6b2f91d45578f7c6cfe20a685bd6802d8fa7ca', 'data/processed/taipei_dataset_b1_pooled.csv': '3b02fb7b338b59f2552972ccb98efa0d723e525826b301cdda021efa9ddd4e50', 'data/processed/taipei_dataset_b1_pooled_schema.json': '63f46e1ae0cba9c19549ee5f8b9ad39084a22f5d2d738b62567b1b3f5e86f340'}
assert sha(PROJECT_ROOT/'data/code/prepare_taipei_dataset_b1.ipynb')=='27ef88b179ae346479cab1d53b35c41a537cf7383d03216d4c4923c32eaa7ff0'
for rel,h in EXPECTED_OUTPUTS.items():assert sha(PROJECT_ROOT/rel)==h,('OUTPUT HASH MISMATCH',rel)
print('四正式人口輸出與producer版本核對通過；不重新處理raw人口')


Python 3.12.2 人口來源／helper hash 11 僅交付鍵檢查來源hash 8
四正式人口輸出與producer版本核對通過；不重新處理raw人口


## Change for Your Own Data

先核對精確路徑及hash，再按schema明示string／nullable型態與__MISSING__；不使用預設NA推論。Raw自身CP950契約與processed不同，此文件不重讀raw作特徵工程。

In [2]:
saved={};contracts={}
for name in u.STEMS:
    contract=json.loads((PROJECT_ROOT/f'data/processed/{name}_schema.json').read_text(encoding='utf-8'))
    assert contract['population_inputs']==POPULATION_INPUTS and contract['handoff_only_inputs']==HANDOFF_ONLY_INPUTS
    saved[name]=u.read_csv(PROJECT_ROOT/f'data/processed/{name}.csv',contract);contracts[name]=contract
base_contract=json.loads((PROJECT_ROOT/'data/processed/taipei_village_base_schema.json').read_text(encoding='utf-8'))
base=u.read_csv(PROJECT_ROOT/'data/processed/taipei_village_base.csv',base_contract)
print('獨立讀saved files之年度／pooled核對',u.validate_bundle(saved,base))
STEM='taipei_dataset_b1_annual'
frame=saved[STEM];contract=contracts[STEM]
assert frame.shape==(912,46)
lookup=base.set_index('village_id')
for col in ['population_code','boundary_code','district_name','population_village_name','boundary_village_name','standard_village_name','verification_status','match_method']:
    assert frame[col].eq(frame.village_id.map(lookup[col])).all(),col
print('代碼／原名稱／標準名／證據狀態與正式里主表完全一致；識別碼維持string')


獨立讀saved files之年度／pooled核對 {'status': 'passed', 'keys': '912 / 456 complete', 'annual_to_pooled_month_counts_and_sums': 'exact', 'fixed_area': 'exact Unit2.2 values'}
代碼／原名稱／標準名／證據狀態與正式里主表完全一致；識別碼維持string


## Definition for each variable

In [3]:
dictionary=pd.DataFrame(contract['columns'])
u.show('Definition for each variable：完整資料字典',dictionary,block=16)
assert list(frame)==dictionary['name'].tolist()
assert [str(frame[c].dtype) for c in frame]==dictionary['dtype'].tolist()
u.show('既有年齡群定義，包含100歲以上終端欄',pd.DataFrame([{'group':k,'source_columns':json.dumps(v,ensure_ascii=False),'count':len(v)} for k,v in contract['age_groups'].items()]))


**Definition for each variable：完整資料字典**

,name,dtype,definition,unit,source,missing_rule
0,village_id,string,共同固定10碼CPTVID，不由人口碼截碼產生,identifier,existing village_crosswalk,不可缺失、不轉數值
1,year,Int64,西元年，只由原民國113／114加1911一次,year,raw 年份,不可缺失
2,district_name,string,沿用既有crosswalk及正式里主表之district_name,not applicable,Unit1.3 crosswalk / Unit2.2 base,不可缺失；原名稱不覆寫
3,standard_village_name,string,沿用既有crosswalk及正式里主表之standard_village_name,not applicable,Unit1.3 crosswalk / Unit2.2 base,不可缺失；原名稱不覆寫
4,population_code,string,沿用既有crosswalk及正式里主表之population_code,identifier,Unit1.3 crosswalk / Unit2.2 base,不可缺失；原名稱不覆寫
5,boundary_code,string,沿用既有crosswalk及正式里主表之boundary_code,identifier,Unit1.3 crosswalk / Unit2.2 base,不可缺失；原名稱不覆寫
6,population_village_name,string,人口原始里名保留，包括糖?里,not applicable,Unit1.3 crosswalk / Unit2.2 base,不可缺失；原名稱不覆寫
7,boundary_village_name,string,固定界原始里名保留，包括糖蔀里,not applicable,Unit1.3 crosswalk / Unit2.2 base,不可缺失；原名稱不覆寫
8,verification_status,string,原證據狀態保留；糖廍provisional不是排除／設NA規則,not applicable,Unit1.3 crosswalk / Unit2.2 base,不可缺失；原名稱不覆寫
9,match_method,string,沿用既有crosswalk及正式里主表之match_method,not applicable,Unit1.3 crosswalk / Unit2.2 base,不可缺失；原名稱不覆寫


,name,dtype,definition,unit,source,missing_rule
16,fixed_village_area_km2,Float64,Unit2.2正式固定EPSG:3826幾何面積；不重新算面積、不用AREA或歷史面積,km²,Unit2.2 village_area_km2,正值不可缺失
17,expected_month_count,Int64,完整年度12或兩年24個月,month,Unit2.4 deterministic monthly aggregation,依欄規則保留NA，不插補
18,observed_month_count,Int64,實際唯一計人口月紀錄數,month,raw year_month / sex 計,缺月以紀錄數反映，不補人口
19,missing_month_count,Int64,應有月份減實際月份數,month,Unit2.4 deterministic monthly aggregation,依欄規則保留NA，不插補
20,valid_total_population_month_count,Int64,total人口各自有效月份數,month,Unit2.4 deterministic monthly aggregation,依欄規則保留NA，不插補
21,total_population_month_sum,Int64,total有效月人口存量加總，非期間不同個人總數,person-month stock,raw 總計,有部分月缺失僅已知月和；全部缺失NA，不能冒稱完整期間
22,mean_monthly_population,Float64,total人口月存量和／12，須該欄完整月份才提供,person,Unit2.4 deterministic monthly aggregation,缺月或該人口欄不完整則NA；不改除以少數月份
23,population_month_coverage,Float64,total有效月數／應有月數,fraction,Unit2.4 deterministic monthly aggregation,依欄規則保留NA，不插補
24,total_population_complete,boolean,total紀錄及有效月份均完整,not applicable,Unit2.4 deterministic monthly aggregation,依欄規則保留NA，不插補
25,valid_elderly_population_month_count,Int64,elderly人口各自有效月份數,month,Unit2.4 deterministic monthly aggregation,依欄規則保留NA，不插補


,name,dtype,definition,unit,source,missing_rule
32,mean_monthly_children_population,Float64,children人口月存量和／12，須該欄完整月份才提供,person,Unit2.4 deterministic monthly aggregation,缺月或該人口欄不完整則NA；不改除以少數月份
33,children_population_month_coverage,Float64,children有效月數／應有月數,fraction,Unit2.4 deterministic monthly aggregation,依欄規則保留NA，不插補
34,children_population_complete,boolean,children紀錄及有效月份均完整,not applicable,Unit2.4 deterministic monthly aggregation,依欄規則保留NA，不插補
35,monthly_records_complete,boolean,所有應有月份紀錄完整,logical flag,Unit2.4 deterministic monthly aggregation,依欄規則保留NA，不插補
36,observed_month_coverage,Float64,實際月份覆蓋率,fraction,Unit2.4 deterministic monthly aggregation,依欄規則保留NA，不插補
37,population_features_valid,boolean,三核心特徵均有有效值,logical flag,Unit2.4 deterministic monthly aggregation,依欄規則保留NA，不插補
38,elderly_ratio_numerator,Int64,elderly比例之人口月存量分子,person-month stock,Unit2.4 deterministic monthly aggregation,相應月資料不完整或總人口和0則NA
39,elderly_ratio_denominator,Int64,elderly比例之總人口月存量有效分母,person-month stock,Unit2.4 deterministic monthly aggregation,相應月資料不完整或總人口和0則NA
40,elderly_ratio,Float64,65歲以上（含100歲以上）人口月和／總人口月和，不平均月／年度比例,fraction,Unit2.4 deterministic monthly aggregation,相關月資料不完整或總人口分母0則NA
41,children_ratio_numerator,Int64,children比例之人口月存量分子,person-month stock,Unit2.4 deterministic monthly aggregation,相應月資料不完整或總人口和0則NA


**既有年齡群定義，包含100歲以上終端欄**

,group,source_columns,count
0,children,"[""0歲數量"", ""1歲數量"", ""2歲數量"", ""3歲數量"", ""4歲數量"", ""5歲數量...",15
1,working_age,"[""15歲數量"", ""16歲數量"", ""17歲數量"", ""18歲數量"", ""19歲數量"", ...",50
2,elderly,"[""65歲數量"", ""66歲數量"", ""67歲數量"", ""68歲數量"", ""69歲數量"", ...",36


## Summary

### Sample

In [4]:
u.show('實際前10列：地理／期間及來源欄',frame.iloc[:10,:18])
u.show('實際前10列：人口核心與完整性',frame.iloc[:10][['village_id','observed_month_count','valid_total_population_month_count','mean_monthly_population','mean_monthly_elderly_population','mean_monthly_children_population','population_density','elderly_ratio','children_ratio']])


**實際前10列：地理／期間及來源欄**

,village_id,year,district_name,standard_village_name,population_code,boundary_code,population_village_name,boundary_village_name,verification_status,match_method,population_district_code,population_sex_scope,source_roc_years_json,period_start,period_end,period_years,fixed_village_area_km2,expected_month_count
0,6300100002,2024,松山區,莊敬里,63000010002,6300100002,莊敬里,莊敬里,verified_exact_match,district_and_village_exact,63000010,計,"[""113""]",2024-01,2024-12,1,1.281723,12
1,6300100002,2025,松山區,莊敬里,63000010002,6300100002,莊敬里,莊敬里,verified_exact_match,district_and_village_exact,63000010,計,"[""114""]",2025-01,2025-12,1,1.281723,12
2,6300100003,2024,松山區,東榮里,63000010003,6300100003,東榮里,東榮里,verified_exact_match,district_and_village_exact,63000010,計,"[""113""]",2024-01,2024-12,1,0.259347,12
3,6300100003,2025,松山區,東榮里,63000010003,6300100003,東榮里,東榮里,verified_exact_match,district_and_village_exact,63000010,計,"[""114""]",2025-01,2025-12,1,0.259347,12
4,6300100004,2024,松山區,三民里,63000010004,6300100004,三民里,三民里,verified_exact_match,district_and_village_exact,63000010,計,"[""113""]",2024-01,2024-12,1,0.134542,12
5,6300100004,2025,松山區,三民里,63000010004,6300100004,三民里,三民里,verified_exact_match,district_and_village_exact,63000010,計,"[""114""]",2025-01,2025-12,1,0.134542,12
6,6300100005,2024,松山區,新益里,63000010005,6300100005,新益里,新益里,verified_exact_match,district_and_village_exact,63000010,計,"[""113""]",2024-01,2024-12,1,0.115138,12
7,6300100005,2025,松山區,新益里,63000010005,6300100005,新益里,新益里,verified_exact_match,district_and_village_exact,63000010,計,"[""114""]",2025-01,2025-12,1,0.115138,12
8,6300100006,2024,松山區,富錦里,63000010006,6300100006,富錦里,富錦里,verified_exact_match,district_and_village_exact,63000010,計,"[""113""]",2024-01,2024-12,1,0.099809,12
9,6300100006,2025,松山區,富錦里,63000010006,6300100006,富錦里,富錦里,verified_exact_match,district_and_village_exact,63000010,計,"[""114""]",2025-01,2025-12,1,0.099809,12


**實際前10列：人口核心與完整性**

,village_id,observed_month_count,valid_total_population_month_count,mean_monthly_population,mean_monthly_elderly_population,mean_monthly_children_population,population_density,elderly_ratio,children_ratio
0,6300100002,12,12,5018.916667,1190.666667,565.5,3915.757309,0.237236,0.112674
1,6300100002,12,12,4884.583333,1188.083333,549.583333,3810.950482,0.243231,0.112514
2,6300100003,12,12,8039.083333,1966.166667,1357.833333,30997.384837,0.244576,0.168904
3,6300100003,12,12,7874.416667,1974.5,1317.5,30362.457218,0.250749,0.167314
4,6300100004,12,12,6591.333333,1752.583333,735.166667,48990.793465,0.265892,0.111535
5,6300100004,12,12,6440.916667,1760.5,677.166667,47872.805422,0.273331,0.105135
6,6300100005,12,12,4387.0,1129.083333,424.666667,38102.243148,0.25737,0.096801
7,6300100005,12,12,4311.5,1150.833333,422.416667,37446.505888,0.266922,0.097974
8,6300100006,12,12,5074.5,1307.25,655.083333,50841.924014,0.257612,0.129093
9,6300100006,12,12,4943.083333,1323.166667,620.5,49525.247262,0.26768,0.125529


### Summary Stats

In [5]:
groups=[(str(y),g) for y,g in frame.groupby('year')] if 'year' in frame else [('2024–2025',frame)]
statistics=[];city=[]
for period,g in groups:
    for feature in u.FEATURES:
        v=g[feature]
        statistics.append({'period':period,'feature':feature,'valid_N':int(v.count()),'missing_N':int(v.isna().sum()),'min':float(v.min()),'q25':float(v.quantile(.25)),'median':float(v.median()),'q75':float(v.quantile(.75)),'max':float(v.max()),'mean_of_villages':float(v.mean()),'std_ddof1':float(v.std())})
    annual='year' in g
    total=int(g[u.sum_col('total',annual)].sum());elderly=int(g[u.sum_col('elderly',annual)].sum());children=int(g[u.sum_col('children',annual)].sum())
    city.append({'period':period,'population_month_sum':total,'elderly_month_sum':elderly,'children_month_sum':children,'mean_monthly_city_population':total/(12 if annual else 24),'city_elderly_ratio':elderly/total,'city_children_ratio':children/total})
u.show('各期三核心特徵完整敘述統計；平均是行政里特徵平均',pd.DataFrame(statistics))
u.show('全市整體比例用人口加總作權重，不平均456里比例',pd.DataFrame(city))


**各期三核心特徵完整敘述統計；平均是行政里特徵平均**

,period,feature,valid_N,missing_N,min,q25,median,q75,max,mean_of_villages,std_ddof1
0,2024,population_density,456,0,52.600089,13501.960384,27251.482971,38931.466181,107146.447060,27574.830458,18113.724744
1,2024,elderly_ratio,456,0,0.132529,0.208489,0.230985,0.251993,0.334525,0.229860,0.034261
2,2024,children_ratio,456,0,0.057276,0.091334,0.109768,0.137924,0.267030,0.118993,0.037680
3,2025,population_density,456,0,51.655697,13141.244306,26748.436955,38326.067128,104127.497798,27043.486822,17717.720615
4,2025,elderly_ratio,456,0,0.140793,0.218539,0.242536,0.263181,0.347787,0.240855,0.034549
5,2025,children_ratio,456,0,0.055211,0.090021,0.108476,0.136774,0.268630,0.117489,0.037853


**全市整體比例用人口加總作權重，不平均456里比例**

,period,population_month_sum,elderly_month_sum,children_month_sum,mean_monthly_city_population,city_elderly_ratio,city_children_ratio
0,2024,30039269,6780091,3677758,2.503272e+06,0.225708,0.122432
1,2025,29476539,6978874,3561360,2.456378e+06,0.236760,0.120820


### 月份、缺失及極端值品質

In [6]:
for period,g in groups:
    u.show(period+'各區完整性',g.groupby('district_name').agg(villages=('village_id','nunique'),min_observed_months=('observed_month_count','min'),max_observed_months=('observed_month_count','max'),min_valid_total=('valid_total_population_month_count','min'),min_valid_elderly=('valid_elderly_population_month_count','min'),min_valid_children=('valid_children_population_month_count','min'),min_coverage=('population_month_coverage','min')).reset_index())
    for feature in u.FEATURES:
        u.show(period+' '+feature+'最高與最低各三里，極端值不刪除',pd.concat([g.nsmallest(3,feature),g.nlargest(3,feature)])[['village_id','district_name','standard_village_name','fixed_village_area_km2','mean_monthly_population',feature]])
u.show('糖廍與士林公館完整保留',frame.loc[frame.village_id.isin(['6300700015','6301100044']),['village_id','population_code','standard_village_name','population_village_name','verification_status','observed_month_count']+u.FEATURES])


**2024各區完整性**

,district_name,villages,min_observed_months,max_observed_months,min_valid_total,min_valid_elderly,min_valid_children,min_coverage
0,中山區,42,12,12,12,12,12,1.0
1,中正區,31,12,12,12,12,12,1.0
2,信義區,41,12,12,12,12,12,1.0
3,內湖區,39,12,12,12,12,12,1.0
4,北投區,42,12,12,12,12,12,1.0
5,南港區,20,12,12,12,12,12,1.0
6,士林區,51,12,12,12,12,12,1.0
7,大同區,25,12,12,12,12,12,1.0
8,大安區,53,12,12,12,12,12,1.0
9,文山區,43,12,12,12,12,12,1.0


**2024 population_density最高與最低各三里，極端值不刪除**

,village_id,district_name,standard_village_name,fixed_village_area_km2,mean_monthly_population,population_density
910,6301200042,北投區,湖田里,16.324434,858.666667,52.600089
818,6301100047,士林區,菁山里,11.720427,1491.083333,127.220901
822,6301100049,士林區,溪山里,10.201475,1466.25,143.729209
518,6300700035,萬華區,凌霄里,0.031744,3401.25,107146.44706
216,6300300037,大安區,群英里,0.068833,6735.583333,97853.558888
516,6300700034,萬華區,忠貞里,0.051465,4473.833333,86929.621765


**2024 elderly_ratio最高與最低各三里，極端值不刪除**

,village_id,district_name,standard_village_name,fixed_village_area_km2,mean_monthly_population,elderly_ratio
600,6300800040,文山區,政大里,2.76735,7986.916667,0.132529
724,6301000040,內湖區,南湖里,0.372824,5882.083333,0.133584
688,6301000022,內湖區,五分里,0.311398,11033.083333,0.133878
162,6300300008,大安區,昌隆里,0.134574,2914.583333,0.334525
158,6300300006,大安區,義村里,0.245901,5087.0,0.31669
516,6300700034,萬華區,忠貞里,0.051465,4473.833333,0.30995


... (8 outputs left out)

**2025 children_ratio最高與最低各三里，極端值不刪除**

,village_id,district_name,standard_village_name,fixed_village_area_km2,mean_monthly_population,children_ratio
907,6301200040,北投區,湖山里,3.751806,1382.583333,0.055211
437,6300600019,大同區,揚雅里,0.065522,4043.166667,0.057752
821,6301100048,士林區,平等里,6.429145,1393.333333,0.05945
381,6300500022,中正區,建國里,0.605006,1825.0,0.26863
379,6300500021,中正區,新營里,0.156612,8460.5,0.241997
367,6300500015,中正區,廈安里,0.08243,4500.583333,0.239728


**糖廍與士林公館完整保留**

,village_id,population_code,standard_village_name,population_village_name,verification_status,observed_month_count,population_density,elderly_ratio,children_ratio
478,6300700015,63000070015,糖廍里,糖?里,provisional_match,12,33436.682753,0.258075,0.07261
479,6300700015,63000070015,糖廍里,糖?里,provisional_match,12,32458.509728,0.265038,0.071374
812,6301100044,63000110044,公館里,公館里,verified_exact_match,12,727.257949,0.256702,0.07978
813,6301100044,63000110044,公館里,公館里,verified_exact_match,12,704.931482,0.284651,0.077369


### 使用範例與後續交付

In [7]:
# 只查交付鍵；不把A/C事故值作為B1輸入，不保存整合資料。
handoff=u.handoff_check(PROJECT_ROOT,saved,HANDOFF_ONLY_INPUTS)
u.show('Unit2.7 many-to-one合併契約',pd.DataFrame(handoff))
if 'year' in frame:
    population_2024=frame.loc[frame.year.eq(2024)]
    population_2025=frame.loc[frame.year.eq(2025)]
    assert len(population_2024)==len(population_2025)==456
    print('跨年度研究保留各自年度；不得以pooled人口取代2024資訊')
else:
    print('pooled供兩年整體描述，不能冒充2024年度資訊')
print('B1為外部環境；未加入事故分群、擬合模型或標準化')


**Unit2.7 many-to-one合併契約**

,dataset,rows_before,rows_after,merge,unmatched,B1_rows,accident_values_read
0,taipei_dataset_a_annual,3648,3648,many_to_one,0,912,False
1,taipei_dataset_a_pooled,1824,1824,many_to_one,0,456,False
2,taipei_dataset_c_annual,3648,3648,many_to_one,0,912,False
3,taipei_dataset_c_pooled,1824,1824,many_to_one,0,456,False


跨年度研究保留各自年度；不得以pooled人口取代2024資訊
B1為外部環境；未加入事故分群、擬合模型或標準化


### Assertions

原始逐月／逐齡及男女人口核對由處理Notebook保存實際結果；本文件獨立驗證正式資料及年度／pooled契約。技術通過仍待人工驗收。

In [8]:
u.validate_dataset(frame,base,True)
assert u.validate_bundle(saved,base)['status']=='passed'
assert frame.monthly_records_complete.all() and frame.population_features_valid.all()
assert frame.population_month_coverage.eq(1).all()
assert frame[u.FEATURES].notna().all().all()
assert 'specification' not in frame
u.verify(PROJECT_ROOT,POPULATION_INPUTS);u.verify(PROJECT_ROOT,HANDOFF_ONLY_INPUTS)
for rel,h in EXPECTED_OUTPUTS.items():assert sha(PROJECT_ROOT/rel)==h
again=u.read_csv(PROJECT_ROOT/'data/processed/taipei_dataset_b1_annual.csv',contract)
pd.testing.assert_frame_equal(again,frame,check_exact=True)
result={'status':'passed','dataset':STEM,'shape':list(frame.shape),'sha256':sha(PROJECT_ROOT/'data/processed/taipei_dataset_b1_annual.csv'),'schema_sha256':sha(PROJECT_ROOT/'data/processed/taipei_dataset_b1_annual_schema.json'),'independent_saved_file_read':True,'bundle_and_handoff':'passed','human_acceptance':'DE / DA / PM pending'}
temp=PROJECT_ROOT/'data/temp/unit24_validation';temp.mkdir(parents=True,exist_ok=True)
(temp/(STEM+'_documentation_summary.json')).write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print(json.dumps(result,ensure_ascii=False,indent=2))

{
  "status": "passed",
  "dataset": "taipei_dataset_b1_annual",
  "shape": [
    912,
    46
  ],
  "sha256": "d4ad76e82ad6f00d5b146c00c0e065d2095cd9fd36fe6bbbad5feb40705ae795",
  "schema_sha256": "89bb5ace9a7abf9bcedcc1ff0d6b2f91d45578f7c6cfe20a685bd6802d8fa7ca",
  "independent_saved_file_read": true,
  "bundle_and_handoff": "passed",
  "human_acceptance": "DE / DA / PM pending"
}